# ESG Report Analyzer v2

End-to-end, deterministic ESG analysis for `.txt` and `.pdf` corporate reports.
Version 2 replaces the human-review checkpoint with conservative automated value
resolution: evidence is localized before extraction, candidates are associated with
the nearest metric label, and ambiguous figures are excluded rather than guessed.

The metric library, scoring rubric, visualizations, and dashboard-generation code are
otherwise retained so dashboard redesign can remain a separate follow-up.


## Cell 1 - Config

Paths, thresholds, and the alias tables that make numeric extraction
tolerant of how different reports phrase units and magnitudes
(`MAGNITUDE_ALIASES`, `UNIT_ALIASES`). `REPORT_PATH` and
`COMPANY_NAME_OVERRIDE` are the two inputs to set per report - `REPORT_PATH`
is read from the `ESG_REPORT_PATH` environment variable so this notebook
doesn't need to be edited per run; `COMPANY_NAME_OVERRIDE` is an escape
hatch for the rare report whose cover page defeats the company-name
heuristic in Cell 3. Grade cutoffs and the certification bonus cap live
here too.

In [ ]:
# ============================================================
# ESG REPORT ANALYZER
# Cell 1 - Imports & Global Configuration
# ============================================================

import os
import re
import json
import math
from pathlib import Path
import numpy as np
import pandas as pd

from rapidfuzz import fuzz

from scoring import MAX_CERTIFICATION_BONUS, METHODOLOGY_VERSION, overall_from_categories, grade_from_score

FUZZY_MATCH_THRESHOLD = 85

# By default, discover every supported report in the reports folder. Setting
# ESG_REPORT_PATH keeps the single-report workflow available for targeted runs.
REPORTS_FOLDER = Path(os.environ.get("ESG_REPORTS_FOLDER", "reports"))
SUPPORTED_REPORT_SUFFIXES = {".txt", ".pdf"}
REPORT_PATHS = sorted(
    path for path in REPORTS_FOLDER.iterdir()
    if path.is_file() and path.suffix.lower() in SUPPORTED_REPORT_SUFFIXES
) if REPORTS_FOLDER.exists() else []

_report_override = os.environ.get("ESG_REPORT_PATH")
if _report_override:
    REPORT_PATH = Path(_report_override)
elif REPORT_PATHS:
    REPORT_PATH = REPORT_PATHS[0]
else:
    raise FileNotFoundError(
        f"No .txt or .pdf reports found in {REPORTS_FOLDER.resolve()}"
    )

RUN_BATCH = (
    not _report_override
    and len(REPORT_PATHS) > 1
    and os.environ.get("ESG_BATCH_CHILD") != "1"
)
print(f"Discovered {len(REPORT_PATHS)} report(s) in {REPORTS_FOLDER.resolve()}")
print(f"Current notebook pass: {REPORT_PATH}")
if RUN_BATCH:
    print("Batch mode enabled; all reports will be evaluated after the analyzer cells.")

# If the automatic company-name heuristic ever guesses wrong on a new
# report, set this manually instead of patching the extraction logic.
COMPANY_NAME_OVERRIDE = None

# ============================================================
# Numeric-extraction tuning
# ============================================================

# How many characters ahead we'll search for a unit when a number isn't
# immediately followed by one (handles PPT/PDF-table text where a value
# and its unit land in different "cells").
NUMERIC_LINK_WINDOW = 80

# Automated resolution: only label-local numeric evidence can become an accepted
# KPI. If two materially different candidates remain equally plausible, the
# analyzer abstains from publishing a primary value for that metric.
MAX_LABEL_VALUE_DISTANCE = 140
COMPETING_LABEL_MARGIN = 12
AUTO_ACCEPT_MARGIN = 10
VALUE_CLUSTER_TOLERANCE = 0.02

# Magnitude suffixes/words that scale a raw number (e.g. "32.31m" -> 32.31e6)
MAGNITUDE_ALIASES = {
    "k": 1e3, "thousand": 1e3,
    "m": 1e6, "mn": 1e6, "million": 1e6,
    "bn": 1e9, "billion": 1e9,
}
MAGNITUDE_PATTERN = r"(?:k|thousand|m(?:n)?|million|bn|billion)"

# Unit aliases: several ways a report might spell/space the same unit
# (e.g. "tCO2e" vs "t CO2e" vs "metric t CO2e"). A rigid single-string
# match misses real disclosures whenever a report uses a variant
# spelling, so each canonical unit maps to a list of tolerant patterns.
UNIT_ALIASES = {
    "tCO2e": [r"t\s*co2e", r"tonnes?\s*co2e", r"metric\s*tonnes?\s*co2e", r"metric\s*t\s*co2e"],
    "kg": [r"kg", r"kilograms?"],
    "tonnes": [r"metric\s*tonnes?", r"tonnes?"],
    "MWh": [r"mwh"],
    "GWh": [r"gwh"],
    "kWh": [r"kwh"],
    "m3": [r"m3", r"m\^3", r"cubic\s*met(?:re|er)s?"],
    "%": [r"%", r"percent"],
    "hours": [r"hours?", r"hrs?"],
    "intensity": [r"(?:kg|t|tonnes?)\s*co2e?\s*(?:/|per)\s*[a-z0-9€$]+"],
}

# Kept for backward compatibility with anything that still expects the
# old flat dict of unit -> single regex string.
UNIT_PATTERNS = {unit: "|".join(aliases) for unit, aliases in UNIT_ALIASES.items()}

# Numbers immediately preceded by one of these words are almost always a
# footnote/scope/category index (e.g. "Scope 3", "Categories 3, 4, 6"),
# not a measured quantity, and should never be linked to a nearby unit.
NUMERIC_LABEL_STOPWORDS = {
    "scope", "category", "categories", "note", "notes", "article",
    "level", "tier", "section", "class", "figure", "table", "appendix",
    "phase", "step", "annex"
}

# Fuzzy-duplicate threshold used to collapse near-identical paragraphs
# (recurring slide headers/footers, repeated summary tables) that exact
# string matching misses.
NEAR_DUPLICATE_THRESHOLD = 93

def normalize_text(text):
    """
    Normalize whitespace while preserving punctuation.
    """

    text = text.replace("\n", " ")
    text = re.sub(r"\s+", " ", text)

    return text.strip()


def split_sentences(text):
    """
    Split report into individual sentences.
    """

    return re.split(r'(?<=[.!?])\s+', text)

## Cell 2 - ESG Metric Library

Configuration objects for every scored metric: `keywords` (fuzzy-matched
against report paragraphs, including common regional/translated
terminology - e.g. "fluctuation" for turnover, two-tier "Supervisory
Board" language for board independence - so the library isn't tuned to
one company's house style), `units`, `weight`, `extract_numeric`
(quantitative vs. qualitative metrics), `polarity`
(`higher_better`/`lower_better`/`neutral`, used to interpret directional
language like "increase"/"reduced" correctly per metric), and `max_score`.
`category` is added automatically when the lookup table is built.

See the docstring in the next cell for the full Disclosure/Performance
scoring rubric.

In [ ]:
# ============================================================
# ESG Metrics Library (Configuration Objects)
# ============================================================

"""
Each metric is stored as a configuration object.

Fields
------
keywords :
    Terms used for fuzzy matching. Covers common regional/translated
    terminology (e.g. German "fluctuation" for turnover, two-tier
    "Supervisory Board" language for board independence) so the
    library isn't only tuned to one company's house style.

units :
    Expected units to search for.

weight :
    Relative importance of the metric.

extract_numeric :
    Whether numerical values should be extracted for this metric.
    Enforced in the evidence-extraction cell (Cell 4) - metrics marked
    False (e.g. Human Rights, Anti-Corruption) never get run through
    numeric extraction, so they can't be assigned a spurious "primary
    KPI value" pulled from an unrelated nearby number.

polarity :
    "higher_better", "lower_better", or "neutral". Used to interpret
    directional language ("increase", "reduced", ...) correctly - the
    same word means opposite things depending on which metric it's
    attached to (e.g. "increase" is good news for Renewable Energy,
    bad news for Scope 1 Emissions), so a single global
    positive/negative word list can't score it correctly.

max_score :
    Maximum metric score.

category :
    Automatically added later when building the lookup table.

Scoring (0-10)

Disclosure (0-5)
----------------
+ Mentioned
+ Specific evidence (numeric value, OR for qualitative metrics: a named
  policy/framework/standard reference)
+ YoY comparison
+ Target disclosed
+ Progress disclosed

Performance (0-5)
-----------------
+2 Net positive, polarity-aware language
+1 Third-party validation / assurance language
+1 Verified numeric trend across two dated values (an objective
   rule-based performance signal, deliberately decoupled from match
   confidence, which measures extraction quality rather than
   company performance)
+1 No negative/incident language at all
"""

# ============================================================
# ESG Metric Configuration
# ============================================================

ESG_METRICS = {

    # ========================================================
    # ENVIRONMENTAL
    # ========================================================

    "Environmental": {

        "Scope 1 Emissions": {
            "keywords": [
                "scope 1", "scope one", "scope1", "direct emissions",
                "direct ghg emissions"
            ],
            "units": ["tCO2e"],
            "weight": 10,
            "extract_numeric": True,
            "polarity": "lower_better",
            "max_score": 10
        },

        "Scope 2 Emissions": {
            "keywords": [
                "scope 2", "scope two", "scope2", "indirect emissions",
                "indirect ghg emissions", "purchased energy emissions"
            ],
            "units": ["tCO2e"],
            "weight": 10,
            "extract_numeric": True,
            "polarity": "lower_better",
            "max_score": 10
        },

        "Scope 3 Emissions": {
            "keywords": [
                "scope 3", "scope three", "scope3", "value chain emissions",
                "upstream emissions", "downstream emissions"
            ],
            "units": ["tCO2e"],
            "weight": 10,
            "extract_numeric": True,
            "polarity": "lower_better",
            "max_score": 10
        },

        "Total GHG Emissions": {
            "keywords": [
                "greenhouse gas emissions", "ghg emissions", "carbon emissions",
                "total emissions", "co2 emissions", "co2e emissions",
                "total ghg", "carbon footprint"
            ],
            "units": ["tCO2e"],
            "weight": 8,
            "extract_numeric": True,
            "polarity": "lower_better",
            "max_score": 10
        },

        "Carbon Intensity": {
            "keywords": [
                "carbon intensity", "emissions intensity", "co2 intensity",
                "ghg intensity", "emissions per revenue", "emissions per shipment"
            ],
            "units": ["intensity"],
            "weight": 8,
            "extract_numeric": True,
            "polarity": "lower_better",
            "max_score": 10
        },

        "Energy Consumption": {
            "keywords": [
                "energy consumption", "energy use", "electricity consumption",
                "total energy", "energy usage", "power consumption"
            ],
            "units": ["MWh", "GWh", "kWh"],
            "weight": 7,
            "extract_numeric": True,
            "polarity": "lower_better",
            "max_score": 10
        },

        "Renewable Energy": {
            "keywords": [
                "renewable energy", "renewable electricity", "green electricity",
                "clean energy", "renewable power", "re100"
            ],
            "units": ["%"],
            "weight": 8,
            "extract_numeric": True,
            "polarity": "higher_better",
            "max_score": 10
        },

        "Water Withdrawal": {
            "keywords": [
                "water withdrawal", "water usage", "water consumption",
                "freshwater withdrawal", "water intake"
            ],
            "units": ["m3"],
            "weight": 7,
            "extract_numeric": True,
            "polarity": "lower_better",
            "max_score": 10
        },

        "Waste Generated": {
            "keywords": [
                "waste generated", "total waste", "waste production",
                "waste footprint"
            ],
            "units": ["tonnes"],
            "weight": 6,
            "extract_numeric": True,
            "polarity": "lower_better",
            "max_score": 10
        },

        "Waste Recycled": {
            "keywords": [
                "recycled waste", "waste recycled", "recycling rate",
                "material recovery", "waste diversion"
            ],
            "units": ["%", "tonnes"],
            "weight": 6,
            "extract_numeric": True,
            "polarity": "higher_better",
            "max_score": 10
        },

        "Circular Economy": {
            "keywords": [
                "circular economy", "resource efficiency", "closed loop",
                "material circularity", "reuse and recycling"
            ],
            "units": [],
            "weight": 5,
            "extract_numeric": False,
            "polarity": "neutral",
            "max_score": 10
        },

        "Biodiversity": {
            "keywords": [
                "biodiversity", "ecosystem", "habitat restoration",
                "nature positive", "species protection", "reforestation",
                "afforestation"
            ],
            "units": [],
            "weight": 5,
            "extract_numeric": False,
            "polarity": "neutral",
            "max_score": 10
        }

    },

    # ========================================================
    # SOCIAL
    # ========================================================

    "Social": {

        "Employee Turnover": {
            "keywords": [
                "employee turnover", "attrition rate", "staff turnover",
                "turnover rate", "fluctuation rate", "total fluctuation",
                "unplanned fluctuation", "staff attrition", "employee fluctuation"
            ],
            "units": ["%"],
            "weight": 7,
            "extract_numeric": True,
            "polarity": "lower_better",
            "max_score": 10
        },

        "Training Hours": {
            "keywords": [
                "training hours", "employee training", "hours of training",
                "learning hours", "hours used for training",
                "training and development", "hours for parents"
            ],
            "units": ["hours"],
            "weight": 6,
            "extract_numeric": True,
            "polarity": "higher_better",
            "max_score": 10
        },

        "Gender Diversity": {
            "keywords": [
                "gender diversity", "female employees", "women employees",
                "share of women", "women in the workforce", "gender balance",
                "proportion of women", "female representation", "women employed",
                "female"
            ],
            "units": ["%"],
            "weight": 8,
            "extract_numeric": True,
            "polarity": "higher_better",
            "max_score": 10
        },

        "Women in Leadership": {
            "keywords": [
                "women in management", "female executives", "female leadership",
                "women in leadership", "female managers",
                "women in senior management", "female representation in management",
                "top management level", "female employees at top management"
            ],
            "units": ["%"],
            "weight": 8,
            "extract_numeric": True,
            "polarity": "higher_better",
            "max_score": 10
        },

        "Employee Engagement": {
            "keywords": [
                "employee engagement", "engagement survey", "employee satisfaction",
                "engagement score", "staff engagement"
            ],
            "units": ["%"],
            "weight": 6,
            "extract_numeric": True,
            "polarity": "higher_better",
            "max_score": 10
        },

        "LTIFR": {
            "keywords": [
                "ltifr", "lost time injury frequency rate",
                "lost-time injury rate", "lost time injury rate"
            ],
            "units": [],
            "weight": 9,
            "extract_numeric": True,
            "polarity": "lower_better",
            "max_score": 10
        },

        "TRIR": {
            "keywords": [
                "trir", "total recordable incident rate",
                "recordable injury rate", "osha recordable rate"
            ],
            "units": [],
            "weight": 9,
            "extract_numeric": True,
            "polarity": "lower_better",
            "max_score": 10
        },

        "Human Rights": {
            "keywords": [
                "human rights", "forced labour", "forced labor",
                "child labour", "child labor", "modern slavery",
                "human rights due diligence"
            ],
            "units": [],
            "weight": 8,
            "extract_numeric": False,
            "polarity": "neutral",
            "max_score": 10
        },

        "Supplier Audits": {
            "keywords": [
                "supplier audit", "supplier assessment", "supplier monitoring",
                "supplier screening", "vendor audit", "supply chain audit"
            ],
            "units": ["%"],
            "weight": 6,
            "extract_numeric": True,
            "polarity": "higher_better",
            "max_score": 10
        },

        "Pay Gap": {
            "keywords": [
                "gender pay gap", "pay equity", "equal pay", "pay parity",
                "compensation gap"
            ],
            "units": ["%"],
            "weight": 8,
            "extract_numeric": True,
            "polarity": "lower_better",
            "max_score": 10
        }

    },

    # ========================================================
    # GOVERNANCE
    # ========================================================

    "Governance": {

        "Independent Directors": {
            "keywords": [
                "independent board", "independent director",
                "non-executive director", "independent non-executive",
                "board independence",
                "independent members of the supervisory board",
                "independent supervisory board members"
            ],
            "units": ["%"],
            "weight": 8,
            "extract_numeric": True,
            "polarity": "higher_better",
            "max_score": 10
        },

        "Board Diversity": {
            "keywords": [
                "board diversity", "female directors", "women on the board",
                "board gender diversity", "diversity of the board",
                "female supervisory board members",
                "women in the supervisory board", "female board of management",
                "supervisory board"
            ],
            "units": ["%"],
            "weight": 8,
            "extract_numeric": True,
            "polarity": "higher_better",
            "max_score": 10
        },

        "Executive Compensation": {
            "keywords": [
                "executive compensation", "executive remuneration",
                "executive pay", "management board remuneration", "ceo pay",
                "remuneration of the board of management"
            ],
            "units": [],
            "weight": 6,
            "extract_numeric": False,
            "polarity": "neutral",
            "max_score": 10
        },

        "Anti-Corruption": {
            "keywords": [
                "anti corruption", "anti-corruption", "bribery",
                "corruption prevention", "anti-bribery"
            ],
            "units": [],
            "weight": 9,
            "extract_numeric": False,
            "polarity": "neutral",
            "max_score": 10
        },

        "Whistleblower Policy": {
            "keywords": [
                "whistleblower", "speak up policy", "speak-up policy",
                "reporting hotline", "ethics hotline", "whistleblowing"
            ],
            "units": [],
            "weight": 6,
            "extract_numeric": False,
            "polarity": "neutral",
            "max_score": 10
        },

        "Cybersecurity": {
            "keywords": [
                "cybersecurity", "cyber security", "information security",
                "data security", "it security"
            ],
            "units": [],
            "weight": 8,
            "extract_numeric": False,
            "polarity": "neutral",
            "max_score": 10
        },

        "Data Privacy": {
            "keywords": [
                "data privacy", "gdpr", "privacy policy", "data protection"
            ],
            "units": [],
            "weight": 8,
            "extract_numeric": False,
            "polarity": "neutral",
            "max_score": 10
        },

        "Ethics": {
            "keywords": [
                "business ethics", "code of ethics", "ethical conduct",
                "code of conduct"
            ],
            "units": [],
            "weight": 7,
            "extract_numeric": False,
            "polarity": "neutral",
            "max_score": 10
        }

    }

}

# ============================================================
# Automatically Build Flat Metric Lookup
# ============================================================

METRIC_LIBRARY = {}

for category, metrics in ESG_METRICS.items():

    for metric_name, config in metrics.items():

        cfg = config.copy()
        cfg["category"] = category
        cfg["name"] = metric_name

        METRIC_LIBRARY[metric_name] = cfg

# ============================================================
# Certification List
# ============================================================

CERTIFICATIONS = [
    "ISO 14001",
    "ISO 45001",
    "ISO 50001",
    "ISO 9001",
    "LEED",
    "B Corp",
    "EcoVadis",
    "CDP",
    "SBTi",
    "RE100",
    "FSC",
    "PEFC",
    "Fairtrade",
    "SA8000",
    "Sedex",
    "UN Global Compact"
]

# Used by the (now-symmetric) qualitative "specific evidence" disclosure
# check - a policy/framework/standard reference substitutes for "has a
# number" on metrics that are inherently non-numeric.
POLICY_REFERENCE_WORDS = [
    "policy", "framework", "standard", "code of conduct", "procedure",
    "guideline", "declaration", "principles", "charter"
] + [c.lower() for c in CERTIFICATIONS]

# ============================================================
# Directional / Sentiment Word Sets
# ============================================================

# A single global POSITIVE_WORDS / NEGATIVE_WORDS list would hardcode a
# word like "increase" as negative even when it describes something
# the company wants to increase (e.g. "increase the share of women in
# management"). Direction words are instead interpreted through each
# metric's `polarity`; only words that are unambiguously good/bad
# regardless of metric stay in a flat list.

DIRECTION_INCREASE_WORDS = [
    "increase", "increased", "increasing", "rose", "rising", "grew",
    "growth", "higher", "up from", "more than"
]

DIRECTION_DECREASE_WORDS = [
    "decrease", "decreased", "decreasing", "reduce", "reduced", "reduction",
    "declined", "declining", "fell", "lower", "down from", "less than"
]

ABSOLUTE_POSITIVE_WORDS = [
    "achieved", "surpassed", "certified", "validated", "science based",
    "net zero", "improved", "on track", "exceeded"
]

ABSOLUTE_NEGATIVE_WORDS = [
    "incident", "spill", "breach", "fatality", "lawsuit", "fine",
    "penalty", "violation", "non-compliance", "non compliance", "misconduct"
]

NEGATION_CUES = [
    "not ", "no ", "without ", "fails to", "failed to", "did not",
    "didn't", "never ", "unable to"
]

# Kept so any external code that still imports the old names doesn't
# break; internally the polarity-aware version below is what's used.
POSITIVE_WORDS = ABSOLUTE_POSITIVE_WORDS + DIRECTION_DECREASE_WORDS
NEGATIVE_WORDS = ABSOLUTE_NEGATIVE_WORDS + DIRECTION_INCREASE_WORDS

print(f"Loaded {len(METRIC_LIBRARY)} ESG metrics.")
print(f"Loaded {len(CERTIFICATIONS)} certifications.")

## Cell 3 - Load & preprocess

Loads `.txt` or `.pdf` input, preserves page boundaries, and builds small evidence
segments from table columns, lines, and sentences. Numeric extraction operates on
these local segments rather than whole PDF pages or presentation slides. PDF input
uses `pypdf` (install with `pip install pypdf` in the notebook kernel if needed).

In [ ]:
# ============================================================
# Cell 3 - Load & Preprocess Sustainability Report
# ============================================================

import re
import shutil
import subprocess
import sys
from pathlib import Path

def _extract_pdf_with_external_python(filepath):
    """Use Codex's bundled document runtime when the notebook kernel lacks PDF crypto."""
    candidates = []
    configured = os.environ.get("ESG_PDF_PYTHON")
    if configured:
        candidates.append(Path(configured))
    candidates.append(
        Path.home() / ".cache/codex-runtimes/codex-primary-runtime/dependencies/python/bin/python3"
    )
    helper = (
        "from pypdf import PdfReader; import sys; "
        "r=PdfReader(sys.argv[1]); "
        "sys.stdout.write('\\n\\n\\f\\n\\n'.join((p.extract_text() or '') for p in r.pages))"
    )
    for python_path in candidates:
        if not python_path.exists() or python_path.resolve() == Path(sys.executable).resolve():
            continue
        completed = subprocess.run(
            [str(python_path), "-c", helper, str(filepath)],
            capture_output=True, text=True, encoding="utf-8", errors="ignore"
        )
        if completed.returncode == 0 and completed.stdout.strip():
            return completed.stdout
    return None

def load_report(filepath):
    """Read a UTF-8 text export or extract all pages from a PDF."""
    filepath = Path(filepath)
    if not filepath.exists():
        raise FileNotFoundError(f"Report not found:\n{filepath.resolve()}")
    if filepath.suffix.lower() == ".pdf":
        try:
            from pypdf import PdfReader
            pages = [(page.extract_text() or "") for page in PdfReader(str(filepath)).pages]
            return "\n\n\f\n\n".join(pages)
        except Exception as first_error:
            pdftotext = shutil.which("pdftotext")
            if pdftotext:
                completed = subprocess.run(
                    [pdftotext, "-layout", str(filepath), "-"],
                    capture_output=True, text=True, encoding="utf-8", errors="ignore"
                )
                if completed.returncode == 0 and completed.stdout.strip():
                    return completed.stdout
            external_text = _extract_pdf_with_external_python(filepath)
            if external_text:
                return external_text
            raise RuntimeError(
                "Could not extract this PDF. Install encrypted-PDF support in the "
                "notebook kernel with: pip install 'pypdf[crypto]'"
            ) from first_error
    return filepath.read_text(encoding="utf-8", errors="ignore")

def clean_text(text):
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = text.replace("\x0c", "\n\n\f\n\n")
    text = text.translate(str.maketrans({"₀":"0", "₁":"1", "₂":"2", "₃":"3"}))
    text = text.replace("\t", "   ")
    text = re.sub(r"[ ]+$", "", text, flags=re.MULTILINE)
    text = re.sub(r"\n{4,}", "\n\n", text)
    return text.strip()

def _normalized_fragment(text):
    return re.sub(r"\s+", " ", text).strip(" |")

def build_evidence_segments(text):
    """Return label-local chunks suitable for metric/value association.

    Runs of three or more spaces are treated as PDF/PPT column boundaries. We
    retain each table cell/line fragment and prose sentences, but intentionally
    do not retain whole pages: page-sized evidence caused cross-metric leakage.
    """
    candidates = []
    pages = text.split("\f")
    for page_no, page in enumerate(pages, start=1):
        lines = [line.strip() for line in page.splitlines() if line.strip()]
        for line in lines:
            columns = [_normalized_fragment(x) for x in re.split(r"\s{3,}", line)]
            columns = [x for x in columns if len(x) >= 12]
            if len(columns) > 1:
                candidates.extend((page_no, "table_cell", x) for x in columns)
            else:
                normalized = _normalized_fragment(line)
                if len(normalized) >= 12:
                    candidates.append((page_no, "line", normalized))

        prose = _normalized_fragment(" ".join(lines))
        for sentence in re.split(r"(?<=[.!?])\s+(?=[A-Z0-9])", prose):
            sentence = sentence.strip()
            if 25 <= len(sentence) <= 350:
                candidates.append((page_no, "sentence", sentence))

    # Exact dedupe first; near-duplicate page furniture is handled below.
    seen, unique = set(), []
    for page_no, kind, segment in candidates:
        key = segment.casefold()
        if key not in seen:
            seen.add(key)
            unique.append({"page": page_no, "kind": kind, "text": segment})
    return unique

def split_sentences(text):
    flat = _normalized_fragment(text.replace("\f", " "))
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", flat) if s.strip()]

COMPANY_SUFFIXES = [
    "Group", "AG", "SE", "PLC", "Inc\\.?", "Corporation", "Corp\\.?",
    "GmbH", "N\\.V\\.", "NV", "Ltd\\.?", "LLC", "Holdings", "A/S",
    "S\\.A\\.", "S\\.p\\.A\\."
]
COMPANY_NAME_PATTERN = re.compile(
    r"^([A-Z][\w&.,'/-]*(?:\s+[A-Z][\w&.,'/-]*){0,5}\s+(?:"
    + "|".join(COMPANY_SUFFIXES) + r"))\b"
)
COMPANY_NAME_EXCLUDE_PATTERNS = [r"^fiscal year", r"^annual report", r"^\d{4}$", r"^page\s*\d+", r"^content", r"^fact sheet"]

def extract_company_name(text, override=None):
    if override:
        return override
    candidates = []
    for raw in text.splitlines()[:80]:
        line = _normalized_fragment(raw)
        if not (4 <= len(line) <= 100):
            continue
        low = line.lower()
        if any(re.match(pat, low) for pat in COMPANY_NAME_EXCLUDE_PATTERNS):
            continue
        if "sustainability" in low or "annual report" in low:
            continue
        candidates.append(line)
    for line in candidates:
        match = COMPANY_NAME_PATTERN.match(line)
        if match:
            return match.group(1).strip()
    anywhere = re.compile(r"([A-Z][A-Za-z&.'/-]+(?:\s+[A-Z][A-Za-z&.'/-]+){0,4}\s+(?:Group|AG|SE|PLC|GmbH|Holdings|A/S))")
    mentions = []
    for line in text.splitlines()[:250]:
        mentions.extend(m.group(1).rstrip("'s") for m in anywhere.finditer(line))
    if mentions:
        return max(set(mentions), key=mentions.count)
    # Filename is a safer fallback than an address/header fragment. Normalize
    # separators/CamelCase and strip report-type suffixes without company maps.
    stem = re.sub(r"(?<=[a-z])(?=[A-Z])", " ", REPORT_PATH.stem)
    stem = re.sub(r"[-_]+", " ", stem)
    stem = re.sub(r"\b(?:annual|sustainability|climate related financial risk|tcfd|factsheet|interactive|low res|report|sr\s*\d{2})\b.*", "", stem, flags=re.I)
    stem = re.sub(r"\b20\d{2}\b|\b\d{2}\b", "", stem)
    stem = re.sub(r"\s+", " ", stem).strip()
    return (stem.title() if stem.islower() else stem) or "Unknown Company"

from table_extraction import build_table_segments, is_combined_scope_aggregate

raw_report = load_report(REPORT_PATH)
clean_report = clean_text(raw_report)
report_year = max([int(y) for y in re.findall(r"20[0-3]\d", REPORT_PATH.name)] or [0]) or None
segments = build_evidence_segments(clean_report)
segments.extend(build_table_segments(
    clean_report, report_year,
    {name: cfg["keywords"] for name, cfg in METRIC_LIBRARY.items()},
    {name: cfg.get("units", []) for name, cfg in METRIC_LIBRARY.items()},
))
sentences = split_sentences(clean_report)
company_name = extract_company_name(clean_report, override=COMPANY_NAME_OVERRIDE)

REPORT = {
    "company": company_name,
    "raw_text": raw_report,
    "clean_text": clean_report,
    "paragraphs": [x["text"] for x in segments],  # backward-compatible alias
    "segments": segments,
    "sentences": sentences,
    "word_count": len(clean_report.split()),
    "sentence_count": len(sentences),
    "paragraph_count": len(segments),
    "report_year": report_year,
}

print("=" * 60)
print("REPORT LOADED")
print("=" * 60)
print(f"Company:      {REPORT['company']}")
print(f"Words:        {REPORT['word_count']:,}")
print(f"Sentences:    {REPORT['sentence_count']}")
print(f"Local evidence segments: {REPORT['paragraph_count']}")
print("\nPreview:\n", _normalized_fragment(clean_report[:800]))


## Cell 4 - Label-local evidence extraction

Matches each metric against small evidence segments. Numeric candidates are limited
to the metric's configured units and must be close to that metric's label. A value is
discarded when another metric label is clearly closer. Proximity-linked numbers are
kept only as diagnostic candidates; the automated resolver never publishes them.

In [ ]:
# ============================================================
# Cell 4 - Label-local ESG Evidence Extraction
# ============================================================

from rapidfuzz import fuzz
import pandas as pd
import re

NUMBER_PATTERN = r"(\d{1,3}(?:,\d{3})+(?:\.\d+)?|\d+(?:\.\d+)?)"
YEAR_PATTERN = re.compile(r"\b(20[0-3]\d)\b")

def _parse_value(raw, magnitude=None):
    value = float(raw.replace(",", ""))
    if magnitude:
        value *= MAGNITUDE_ALIASES.get(magnitude.lower(), 1)
    return value

def _span_distance(a, b):
    if a[1] < b[0]: return b[0] - a[1]
    if b[1] < a[0]: return a[0] - b[1]
    return 0

def _literal_label_spans(text, metric_name):
    low = text.lower()
    spans = []
    for keyword in METRIC_LIBRARY[metric_name]["keywords"]:
        start = 0
        keyword_low = keyword.lower()
        if metric_name == "Total GHG Emissions" and not ("total" in keyword_low or "footprint" in keyword_low):
            continue
        while True:
            pos = low.find(keyword_low, start)
            if pos < 0: break
            prefix = low[max(0, pos-5):pos]
            if prefix.endswith("non-") or prefix.endswith("not "):
                start = pos + max(1, len(keyword_low))
                continue
            spans.append((pos, pos + len(keyword_low), keyword))
            start = pos + max(1, len(keyword_low))
    return spans

def _nearest_year(text, span):
    years = [(m.group(1), _span_distance(span, m.span())) for m in YEAR_PATTERN.finditer(text)]
    if not years:
        return None
    ranked = sorted(years, key=lambda x: x[1])
    nearest = ranked[0]
    # Multi-year prose is usable only when one year is materially closer.
    # Flattened table series are handled separately by build_table_segments.
    if len({year for year, _ in years}) > 1 and len(ranked) > 1 and ranked[1][1] - nearest[1] < 12:
        return None
    return nearest[0] if nearest[1] <= 80 else None

def _is_target_context(text, span):
    context = text[max(0, span[0]-80):min(len(text), span[1]+80)].lower()
    target_words = ("target", "goal", "by 20", "ambition", "at least", "no more than", "≤", "≥")
    target_verbs = re.search(r"\b(reduce|achieve|maintain|reach|realize|increase)\b", context)
    return any(word in context for word in target_words) or bool(target_verbs)

def _is_actual_context(text, span):
    context = text[max(0, span[0]-100):min(len(text), span[1]+100)].lower()
    return bool(re.search(r"\b(amounted to|was|were|reached|remained|reported|result|as of|totalled|total)\b", context))

def _is_subcomponent_context(text, span, metric_name):
    context = text[max(0, span[0]-110):min(len(text), span[1]+110)].lower()
    if re.search(r"\b(of which|including|resulting from|subset|biogenic|regional split|these emissions|avoided|avoidance)\b", context):
        return True
    if re.search(r"\b(additional|reduction|decrease|increase)\b.{0,45}\b(scope|emissions?)\b", context):
        return True  # change amount, not the absolute KPI
    if metric_name == "Energy Consumption" and re.search(r"\b(?:(?:non[- ]?fossil|fossil|renewable|non[- ]?renewable) energy consumption|electricity consumption)\b", context):
        return True
    return False

def _competing_label_distance(text, metric_name, value_span):
    distances = []
    if metric_name in ("Scope 1 Emissions", "Scope 2 Emissions"):
        for match in re.finditer(r"scope\s*1\s*(?:&|and)\s*2", text, re.I):
            distances.append(_span_distance(value_span, match.span()))
    for other in METRIC_LIBRARY:
        if other == metric_name:
            continue
        for start, end, _ in _literal_label_spans(text, other):
            distances.append(_span_distance(value_span, (start, end)))
    return min(distances) if distances else 10**9

def _accounting_variant(text):
    low = text.lower().replace("–", "-").replace("−", "-")
    if re.search(r"market[- ]based", low): return "market-based"
    if re.search(r"location[- ]based", low): return "location-based"
    return "default"

def extract_numeric_values(text, metric_name, segment=None):
    """Extract metric-specific candidates; never scan unrelated units."""
    if segment and segment.get("kind") == "structured_table_cell":
        if metric_name not in segment.get("matched_metrics", []):
            return []
        span = (0, len(text))
        return [{
            "value": segment["table_value"], "unit": segment["table_unit"],
            "span": span, "source": "structured_table_cell",
            "year": segment.get("table_year"), "year_source": segment.get("year_source"),
            "variant": segment.get("variant", "default"), "table_context": True,
            "label_distance": 0, "target_context": False, "actual_context": True,
            "subcomponent_context": _is_subcomponent_context(segment.get("table_label", text), span, metric_name),
        }]
    config = METRIC_LIBRARY[metric_name]
    labels = _literal_label_spans(text, metric_name)
    if not labels:
        return []  # fuzzy evidence may count as disclosure, not numeric proof

    candidates = []
    expected_units = config.get("units", [])
    for unit in expected_units:
        aliases = "|".join(UNIT_ALIASES.get(unit, [re.escape(unit)]))
        pattern = re.compile(rf"(?<![\w.]){NUMBER_PATTERN}\s*(?P<magnitude>{MAGNITUDE_PATTERN})?\s*(?P<matched_unit>{aliases})", re.I)
        for match in pattern.finditer(text):
            open_paren = text.rfind("(", 0, match.start())
            close_before = text.rfind(")", 0, match.start())
            close_after = text.find(")", match.end(), match.end()+8)
            if open_paren > close_before and close_after >= 0:
                continue  # unit/magnitude legend or parenthetical comparison
            magnitude = match.group("magnitude")
            matched_unit = match.group("matched_unit")
            # In US disclosures, uppercase MTCO2e means metric tonnes, not
            # million tonnes. Preserve lowercase MtCO2e as a magnitude.
            if magnitude == "M" and matched_unit.startswith("T"):
                magnitude = None
            value = _parse_value(match.group(1), magnitude)
            if unit == "%" and not (0 <= value <= 100):
                continue
            value_span = match.span()
            if is_combined_scope_aggregate(text, value_span[0], metric_name):
                continue
            nearest_label = min(labels, key=lambda label: _span_distance(value_span, label[:2]))
            if value_span[1] <= nearest_label[0] and re.search(r"[.!?]", text[value_span[1]:nearest_label[0]]):
                continue  # value belongs to the preceding sentence/row
            label_distance = min(_span_distance(value_span, (s, e)) for s, e, _ in labels)
            competitor_distance = _competing_label_distance(text, metric_name, value_span)
            if label_distance > MAX_LABEL_VALUE_DISTANCE:
                continue
            # Require the current metric label to be materially closer. A tie
            # such as "Scope 1 & 2 ... 2.5m tCO2e" is not separable and must
            # not become the KPI for both metrics.
            if competitor_distance < 10**9 and label_distance + COMPETING_LABEL_MARGIN >= competitor_distance:
                continue
            candidates.append({
                "value": value, "unit": unit, "span": value_span,
                "source": "direct", "year": _nearest_year(text, value_span),
                "year_source": "explicit" if _nearest_year(text, value_span) else None,
                "variant": _accounting_variant(text), "table_context": False,
                "label_distance": label_distance,
                "target_context": _is_target_context(text, value_span),
                "actual_context": _is_actual_context(text, value_span),
                "subcomponent_context": _is_subcomponent_context(text, value_span, metric_name),
            })

        # Table rows often put the unit before the current-period value:
        # "Total GHG emissions tonnes CO2e 45,220,681 42,942,008".
        # Skip numeric parenthetical scale headers such as "(1,000 tCO2e)"
        # because their following series needs explicit column mapping.
        unit_pattern = re.compile(rf"(?:(?P<unit_magnitude>{MAGNITUDE_PATTERN})\s*)?(?P<matched_unit>{aliases})", re.I)
        for unit_match in unit_pattern.finditer(text):
            preceding_labels = [(s, e) for s, e, _ in labels if e <= unit_match.start()]
            if not preceding_labels or min(unit_match.start()-e for _, e in preceding_labels) > 60:
                continue
            before = text[max(0, unit_match.start()-18):unit_match.start()]
            if re.search(r"\(\s*[\d,.]+\s*$", before):
                continue
            after = text[unit_match.end():unit_match.end()+45]
            number_match = re.match(rf"\s*\)?\s+{NUMBER_PATTERN}", after)
            if not number_match:
                continue
            raw_value = number_match.group(1)
            if raw_value.isdigit() and 1990 <= int(raw_value) <= 2039:
                continue  # table header year, not a measured value
            unit_magnitude = unit_match.group("unit_magnitude")
            if unit_magnitude == "M" and unit_match.group("matched_unit").startswith("T"):
                unit_magnitude = None
            value = _parse_value(raw_value, unit_magnitude)
            if unit == "%" and not (0 <= value <= 100):
                continue
            start = unit_match.end() + number_match.start(1)
            value_span = (start, start + len(number_match.group(1)))
            if is_combined_scope_aggregate(text, value_span[0], metric_name):
                continue
            label_distance = min(_span_distance(value_span, (s, e)) for s, e, _ in labels)
            competitor_distance = _competing_label_distance(text, metric_name, value_span)
            if label_distance > MAX_LABEL_VALUE_DISTANCE:
                continue
            if competitor_distance < 10**9 and label_distance + COMPETING_LABEL_MARGIN >= competitor_distance:
                continue
            candidates.append({
                "value": value, "unit": unit, "span": value_span,
                "source": "unit_before_table", "year": _nearest_year(text, value_span),
                "year_source": "explicit" if _nearest_year(text, value_span) else None,
                "variant": _accounting_variant(text), "table_context": False,
                "label_distance": label_distance,
                "target_context": _is_target_context(text, value_span),
                "actual_context": _is_actual_context(text, value_span),
                "subcomponent_context": _is_subcomponent_context(text, value_span, metric_name),
            })

    # Unitless rate metrics: require a decimal or a comparator and a very short
    # distance to the literal label. This captures "TRIR below 1.0" and
    # "LTIFR ... 10.8" without attaching page numbers or headcounts.
    if not expected_units and config.get("extract_numeric"):
        num_pat = re.compile(rf"(?:[<>=≤≥]\s*)?{NUMBER_PATTERN}")
        for match in num_pat.finditer(text):
            raw = match.group(1)
            value = float(raw.replace(",", ""))
            if 1990 <= value <= 2039 and raw.isdigit():
                continue
            value_span = match.span()
            distance = min(_span_distance(value_span, (s, e)) for s, e, _ in labels)
            context = text[max(0, match.start()-3):match.end()+3]
            if distance > 55 or ("." not in raw and not re.search(r"[<>=≤≥]", context)):
                continue
            candidates.append({
                "value": value, "unit": "rate", "span": value_span,
                "source": "direct", "year": _nearest_year(text, value_span),
                "year_source": "explicit" if _nearest_year(text, value_span) else None,
                "variant": _accounting_variant(text), "table_context": False,
                "label_distance": distance,
                "target_context": _is_target_context(text, value_span),
                "actual_context": _is_actual_context(text, value_span),
                "subcomponent_context": _is_subcomponent_context(text, value_span, metric_name),
            })

    # Deduplicate candidates within a segment.
    unique = {}
    for item in candidates:
        key = (item["value"], item["unit"], item["year"], item["target_context"])
        if key not in unique or item["label_distance"] < unique[key]["label_distance"]:
            unique[key] = item
    return list(unique.values())

def fuzzy_contains(text, keyword, threshold=FUZZY_MATCH_THRESHOLD):
    text, keyword = text.lower(), keyword.lower()
    if keyword in text:
        return True, 100
    words, keyword_words = text.split(), keyword.split()
    if not keyword_words or len(words) < len(keyword_words):
        return False, 0
    best = 0
    for i in range(len(words) - len(keyword_words) + 1):
        window_words = words[i:i+len(keyword_words)]
        score = fuzz.ratio(" ".join(window_words), keyword)
        if score >= threshold and all(any(fuzz.ratio(k, w) >= 75 for w in window_words) for k in keyword_words):
            best = max(best, score)
    return best >= threshold, best

def _is_negated(text_lower, pos):
    return any(cue in text_lower[max(0, pos-25):pos] for cue in NEGATION_CUES)

def find_sentiment(sentence, polarity="neutral"):
    text = sentence.lower(); positives, negatives = [], []
    def scan(words, bucket):
        for word in words:
            start = 0
            while True:
                pos = text.find(word, start)
                if pos < 0: break
                if not _is_negated(text, pos): bucket.append(word)
                start = pos + len(word)
    scan(ABSOLUTE_POSITIVE_WORDS, positives); scan(ABSOLUTE_NEGATIVE_WORDS, negatives)
    if polarity == "higher_better":
        scan(DIRECTION_INCREASE_WORDS, positives); scan(DIRECTION_DECREASE_WORDS, negatives)
    elif polarity == "lower_better":
        scan(DIRECTION_DECREASE_WORDS, positives); scan(DIRECTION_INCREASE_WORDS, negatives)
    return positives, negatives

def confidence_score(fuzzy_score, values, positives, negatives, segment_kind):
    locality_bonus = 4 if segment_kind in ("table_cell", "line") else 0
    distance_bonus = max([0] + [max(0, 12 - v["label_distance"] / 10) for v in values])
    return min(100, max(0, fuzzy_score + locality_bonus + distance_bonus + len(positives) - len(negatives)))

evidence_rows = []
for metric_name, metric in METRIC_LIBRARY.items():
    for segment in REPORT["segments"]:
        text = segment["text"]
        best_keyword, best_score = None, 0
        for keyword in metric["keywords"]:
            matched, score = fuzzy_contains(text, keyword)
            if matched and score > best_score:
                best_keyword, best_score = keyword, score
        if best_keyword is None:
            continue
        values = extract_numeric_values(text, metric_name, segment) if metric["extract_numeric"] else []
        positives, negatives = find_sentiment(text, metric.get("polarity", "neutral"))
        confidence = confidence_score(best_score, values, positives, negatives, segment["kind"])
        evidence_rows.append({
            "Category": metric["category"], "Metric": metric_name,
            "Keyword": best_keyword, "Confidence": confidence,
            "Paragraph": text, "Numeric Values": values,
            "Positive Words": positives, "Negative Words": negatives,
            "Weight": metric["weight"], "Max Score": metric["max_score"],
            "Polarity": metric.get("polarity", "neutral"),
            "Page": segment["page"], "Segment Kind": segment["kind"],
        })

EVIDENCE_DF = pd.DataFrame(evidence_rows)
if not EVIDENCE_DF.empty:
    EVIDENCE_DF = (EVIDENCE_DF.sort_values("Confidence", ascending=False)
                   .drop_duplicates(subset=["Metric", "Paragraph"])
                   .reset_index(drop=True))
print("=" * 60)
print("LABEL-LOCAL EVIDENCE EXTRACTION COMPLETE")
print("=" * 60)
print(f"Evidence rows: {len(EVIDENCE_DF)}; metrics found: {EVIDENCE_DF['Metric'].nunique() if len(EVIDENCE_DF) else 0}")


## Cell 5 - Candidate data model

Normalizes metric-specific numeric candidates. Candidates retain label distance,
target/actual context, page, and segment type for deterministic resolution.

In [ ]:
# ============================================================
# Cell 5 - Normalize Evidence & Candidate Data Model
# ============================================================

import pandas as pd

value_rows = []
for _, row in EVIDENCE_DF.iterrows():
    for item in row["Numeric Values"] if isinstance(row["Numeric Values"], list) else []:
        value_rows.append({
            "Category": row["Category"], "Metric": row["Metric"],
            "Value": item["value"], "Unit": item["unit"],
            "Source": item.get("source", "direct"), "Year": item.get("year"),
            "Year Source": item.get("year_source"), "Variant": item.get("variant", "default"),
            "Table Context": bool(item.get("table_context", False)),
            "Confidence": row["Confidence"], "Paragraph": row["Paragraph"],
            "Page": row.get("Page"), "Segment Kind": row.get("Segment Kind"),
            "Label Distance": item.get("label_distance", 999),
            "Target Context": bool(item.get("target_context", False)),
            "Actual Context": bool(item.get("actual_context", False)),
            "Subcomponent Context": bool(item.get("subcomponent_context", False)),
        })

CANDIDATE_VALUES_DF = pd.DataFrame(value_rows)
if not CANDIDATE_VALUES_DF.empty:
    CANDIDATE_VALUES_DF = (CANDIDATE_VALUES_DF
        .sort_values(["Confidence", "Label Distance"], ascending=[False, True])
        .drop_duplicates(subset=["Metric", "Value", "Unit", "Year", "Paragraph"])
        .reset_index(drop=True))
    # A line, table cell, and sentence can be three views of the same source
    # passage. Collapse them per page/value so repetition cannot masquerade as
    # independent corroboration; retain the most conservative context flags.
    collapsed = []
    for _, group in CANDIDATE_VALUES_DF.groupby(
        ["Metric", "Value", "Unit", "Year", "Variant", "Page"], dropna=False, sort=False
    ):
        row = group.sort_values(["Label Distance", "Confidence"], ascending=[True, False]).iloc[0].copy()
        row["Confidence"] = group["Confidence"].max()
        row["Target Context"] = group["Target Context"].any()
        row["Actual Context"] = group["Actual Context"].any()
        row["Subcomponent Context"] = group["Subcomponent Context"].any()
        row["Table Context"] = group["Table Context"].any()
        collapsed.append(row)
    CANDIDATE_VALUES_DF = pd.DataFrame(collapsed).reset_index(drop=True)
    report_year = REPORT.get("report_year")
    if report_year:
        future_year = pd.to_numeric(CANDIDATE_VALUES_DF["Year"], errors="coerce") > report_year
        CANDIDATE_VALUES_DF["Target Context"] = CANDIDATE_VALUES_DF["Target Context"] | future_year.fillna(False)

found_certifications = [c for c in CERTIFICATIONS if c.lower() in REPORT["clean_text"].lower()]
CERTIFICATIONS_DF = pd.DataFrame({"Certification": found_certifications})
print(f"Numeric candidates: {len(CANDIDATE_VALUES_DF)}")
print(f"Certifications: {len(CERTIFICATIONS_DF)}")


## Cell 5.1 - Automated value resolution

This replaces the human-review pause. Candidates are ranked using evidence
confidence, label distance, actual-vs-target context, reporting year, and repeated
support. A candidate is accepted only when it wins by a clear margin or materially
agrees with the runner-up. Otherwise the metric is marked `abstained`; ambiguous
figures do not enter scoring or the dashboard.

In [ ]:
# ============================================================
# Cell 5.1 - Automated Value Resolution (no human checkpoint)
# ============================================================

def _relative_difference(a, b):
    return abs(a-b) / max(abs(a), abs(b), 1e-9)

def _candidate_score(row, latest_year, support):
    score = float(row["Confidence"]) - min(float(row["Label Distance"]), 140) * 0.18
    score += min(12, 4 * (support - 1))
    if row["Target Context"]: score -= 18
    if row["Actual Context"]: score += 12
    if row["Subcomponent Context"]: score -= 18
    if row.get("Table Context", False): score += 25
    if pd.notna(row["Year"]) and str(row["Year"]) == str(latest_year): score += 8
    if row["Segment Kind"] in ("table_cell", "line"): score += 3
    return score

accepted, decisions = [], []
if not CANDIDATE_VALUES_DF.empty:
    candidates = CANDIDATE_VALUES_DF.copy()
    detected_years = [int(y) for y in candidates["Year"].dropna().astype(str) if str(y).isdigit()]
    latest_year = REPORT.get("report_year") or (max(detected_years) if detected_years else None)

    # Resolve independently by metric/unit/period. Unknown-period values compete
    # with each other; dated values from different years remain available for YoY.
    candidates["Period"] = candidates["Year"].fillna("Unknown").astype(str)
    for (metric, unit, period, variant), group in candidates.groupby(["Metric", "Unit", "Period", "Variant"], sort=False):
        group = group.copy()
        actual = group[~group["Target Context"]]
        if actual.empty:
            decisions.append({
                "Metric": metric, "Unit": unit, "Period": period, "Variant": variant,
                "Decision": "target_only", "Selected Value": None,
                "Reason": "excluded: numeric target is not current performance",
                "Candidates": len(group),
            })
            continue
        complete = actual[~actual["Subcomponent Context"]]
        if complete.empty:
            decisions.append({
                "Metric": metric, "Unit": unit, "Period": period, "Variant": variant,
                "Decision": "subcomponent_only", "Selected Value": None,
                "Reason": "excluded: only a component, avoided impact, or change amount was found",
                "Candidates": len(actual),
            })
            continue
        pool = complete
        # Count materially agreeing mentions as support for the same figure.
        supports = []
        for _, row in pool.iterrows():
            agreeing = pool[pool["Value"].apply(lambda other: _relative_difference(row["Value"], other) <= VALUE_CLUSTER_TOLERANCE)]
            supports.append(max(1, agreeing["Page"].nunique()))
        pool["Support"] = supports
        pool["Resolution Score"] = [
            _candidate_score(row, latest_year, support)
            for (_, row), support in zip(pool.iterrows(), supports)
        ]
        pool = pool.sort_values(["Resolution Score", "Label Distance"], ascending=[False, True])
        best = pool.iloc[0]
        runner = pool.iloc[1] if len(pool) > 1 else None
        agrees = runner is not None and _relative_difference(best["Value"], runner["Value"]) <= VALUE_CLUSTER_TOLERANCE
        margin = float("inf") if runner is None else best["Resolution Score"] - runner["Resolution Score"]
        is_accepted = runner is None or agrees or margin >= AUTO_ACCEPT_MARGIN
        reason = "single label-local candidate" if runner is None else (
            "corroborated by agreeing evidence" if agrees else (
                f"clear score margin ({margin:.1f})" if is_accepted
                else f"abstained: top candidates differ and margin {margin:.1f} < {AUTO_ACCEPT_MARGIN}"
            )
        )
        decisions.append({
            "Metric": metric, "Unit": unit, "Period": period, "Variant": variant,
            "Decision": "accepted" if is_accepted else "abstained",
            "Selected Value": best["Value"] if is_accepted else None,
            "Reason": reason, "Candidates": len(pool),
        })
        if is_accepted:
            out = best.to_dict()
            out["Resolution Score"] = float(best["Resolution Score"])
            out["Resolution Reason"] = reason
            out["Needs Review"] = False
            out["Review Reason"] = ""
            accepted.append(out)

VALUES_DF = pd.DataFrame(accepted)
VALUE_RESOLUTION_DF = pd.DataFrame(decisions)
VALUE_CONFLICTS_DF = VALUE_RESOLUTION_DF[VALUE_RESOLUTION_DF["Decision"] == "abstained"].copy() if len(VALUE_RESOLUTION_DF) else pd.DataFrame()

# Scope 1/2/3 are mutually distinct disclosures. If automated extraction assigns
# an identical figure to more than one scope, it has almost certainly captured a
# combined Scope 1+2 total; exclude the collision rather than publishing it twice.
if not VALUES_DF.empty:
    scope_names = {"Scope 1 Emissions", "Scope 2 Emissions", "Scope 3 Emissions"}
    scope_mask = VALUES_DF["Metric"].isin(scope_names)
    duplicated_scope = VALUES_DF[scope_mask].duplicated(subset=["Value", "Unit", "Year"], keep=False)
    collision_indices = VALUES_DF[scope_mask].index[duplicated_scope]
    if len(collision_indices):
        collisions = VALUES_DF.loc[collision_indices]
        extra = [{"Metric": row["Metric"], "Unit": row["Unit"], "Period": str(row["Year"] or "Unknown"), "Variant": row.get("Variant", "default"),
                  "Decision": "abstained", "Selected Value": None,
                  "Reason": "abstained: identical figure assigned to multiple GHG scopes",
                  "Candidates": 1} for _, row in collisions.iterrows()]
        VALUE_RESOLUTION_DF = pd.concat([VALUE_RESOLUTION_DF, pd.DataFrame(extra)], ignore_index=True)
        VALUE_CONFLICTS_DF = VALUE_RESOLUTION_DF[VALUE_RESOLUTION_DF["Decision"] == "abstained"].copy()
        VALUES_DF = VALUES_DF.drop(index=collision_indices).reset_index(drop=True)

# Rebuild summaries from accepted data so rejected/ambiguous numbers cannot
# accidentally count as specific disclosure downstream.
summary_rows = []
for metric_name, config in METRIC_LIBRARY.items():
    evidence = EVIDENCE_DF[EVIDENCE_DF["Metric"] == metric_name]
    values = VALUES_DF[VALUES_DF["Metric"] == metric_name] if not VALUES_DF.empty else pd.DataFrame()
    summary_rows.append({
        "Category": config["category"], "Metric": metric_name,
        "Weight": config["weight"], "Max Score": config["max_score"],
        "Evidence Count": len(evidence), "Numeric Values": len(values),
        "Values Needing Review": 0, "Detected": len(evidence) > 0,
        "Highest Confidence": evidence["Confidence"].max() if len(evidence) else 0,
    })
METRIC_SUMMARY_DF = pd.DataFrame(summary_rows)
METRIC_SUMMARY_DF["Disclosure %"] = METRIC_SUMMARY_DF["Detected"].astype(int) * 100
CATEGORY_SUMMARY_DF = (METRIC_SUMMARY_DF.groupby("Category")
    .agg(Metrics=("Metric","count"), Found=("Detected","sum"), AvgConfidence=("Highest Confidence","mean"))
    .reset_index())
CATEGORY_SUMMARY_DF["Disclosure %"] = (CATEGORY_SUMMARY_DF["Found"] / CATEGORY_SUMMARY_DF["Metrics"] * 100).round(1)
ESG_DATA = {"report": REPORT, "evidence": EVIDENCE_DF, "candidates": CANDIDATE_VALUES_DF,
            "values": VALUES_DF, "value_resolution": VALUE_RESOLUTION_DF,
            "metric_summary": METRIC_SUMMARY_DF, "category_summary": CATEGORY_SUMMARY_DF,
            "certifications": CERTIFICATIONS_DF}

print("=" * 70)
print("AUTOMATED VALUE RESOLUTION")
print("=" * 70)
print(f"Accepted values: {len(VALUES_DF)}")
print(f"Abstained groups: {len(VALUE_CONFLICTS_DF)}")
if len(VALUE_CONFLICTS_DF):
    print(VALUE_CONFLICTS_DF[["Metric","Unit","Period","Reason"]].to_string(index=False))


In [ ]:
# Human correction helper removed in v2. Ambiguous values are automatically
# excluded, and VALUE_RESOLUTION_DF records every acceptance/abstention decision.


### Automated audit

There is no manual correction step. Inspect `VALUE_RESOLUTION_DF` for traceability;
the remainder of the notebook runs immediately on accepted values only.

In [ ]:
# No manual corrections required.
print(VALUE_RESOLUTION_DF["Decision"].value_counts() if len(VALUE_RESOLUTION_DF) else "No numeric candidates")


In [ ]:
# Machine-checkable gate: flagged values can never reach scoring.
assert VALUES_DF.empty or not VALUES_DF["Needs Review"].any()
print(f"Automated gate passed: {len(VALUES_DF)} accepted values; {len(VALUE_CONFLICTS_DF)} ambiguous groups excluded.")


## Cell 6 - Metric Scoring Engine

Scores each metric on Disclosure (0-5) and Performance (0-5) against
`EVIDENCE_DF`/`VALUES_DF`.

- **Disclosure's "specific evidence" criterion is symmetric**: a numeric
  value for quantitative metrics, or a named policy/framework/standard
  reference for metrics configured as qualitative - so qualitative
  metrics aren't structurally capped below quantitative ones.
- **YoY detection** combines generic comparison phrases
  (`YOY_PHRASE_WORDS`) with a check for two numeric values of the same
  metric tagged with different years (`has_dated_pair`) - not a list of
  literal year strings, which would silently stop detecting a comparison
  the moment the report covers a different year.
- **`detect_verified_trend()`** is the core Performance signal: it
  requires two dated numeric values for the same metric+unit moving in
  the direction the metric's `polarity` calls "better". This is a
  deliberate choice to keep Performance driven by the actual reported
  numbers rather than by how cleanly a sentence's keywords matched.
- Every sub-criterion (`Disc_Mentioned`, `Disc_Specific`, `Disc_YoY`,
  `Disc_Target`, `Disc_Progress`, `Perf_NetPositive`, `Perf_Validation`,
  `Perf_Trend`, `Perf_Clean`) is stored as its own column, which is what
  makes the Cell 9 per-criterion heatmap possible.

In [ ]:
# ============================================================
# Cell 6 - Metric Scoring Engine
# ============================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# Word Lists
# ------------------------------------------------------------

TARGET_WORDS = [
    "target", "goal", "objective", "commitment", "roadmap", "ambition"
]

# Hardcoding literal year strings ("2023", "2024", ...) as YoY signal
# words breaks the moment the report is from a different year, and
# more importantly doesn't actually detect a year-over-year
# *comparison* - it fires on any paragraph that merely mentions a year
# at all (a cover slide, a target date, a founding year). Generic
# comparison phrases are kept here; the real signal comes from the
# "has two dated values" check in score_metric, driven off the Year
# tag captured during numeric extraction.
YOY_PHRASE_WORDS = [
    "compared to", "year-on-year", "year over year", "vs.", "versus",
    "previous year", "prior year", "baseline year"
]

PROGRESS_WORDS = [
    "progress", "achieved", "completed", "reduced", "improved",
    "decreased", "increased", "on track"
]

VALIDATION_WORDS = [
    "verified", "assured", "externally assured", "validated",
    "science based", "sbti", "third-party", "independently"
]


def contains_any(text, keywords):
    text = text.lower()
    return any(word.lower() in text for word in keywords)


# ------------------------------------------------------------
# Verified numeric trend
# ------------------------------------------------------------
#
# A confidence-based proxy (e.g. mean keyword-match confidence) rewards
# clean keyword-matching, not good ESG performance - a perfectly-matched
# sentence describing a fine or a missed target would score just as
# easily as one describing real progress. This instead looks for two
# numeric values of the same metric+unit tagged with different years
# and checks whether the change moved in the direction the metric's
# polarity calls "better". It's still entirely rule-based (no sentiment
# words involved) and is the closest thing to an objective performance
# signal the text supports.

def detect_verified_trend(metric_name, config):

    if not config["extract_numeric"] or VALUES_DF.empty:
        return False

    vals = VALUES_DF[
        (VALUES_DF["Metric"] == metric_name)
        & (VALUES_DF["Year"].notna())
        & (~VALUES_DF["Target Context"])
    ]

    if len(vals) < 2:
        return False

    for unit, group in vals.groupby("Unit"):

        by_year = (
            group
            .sort_values("Confidence", ascending=False)
            .drop_duplicates(subset="Year")
            .sort_values("Year")
        )

        if by_year["Year"].nunique() < 2:
            continue

        first_val = by_year.iloc[0]["Value"]
        last_val = by_year.iloc[-1]["Value"]

        if config["polarity"] == "higher_better" and last_val > first_val:
            return True
        if config["polarity"] == "lower_better" and last_val < first_val:
            return True

    return False


# ------------------------------------------------------------
# Score Individual Metric
# ------------------------------------------------------------

def score_metric(metric_name):

    evidence = EVIDENCE_DF[EVIDENCE_DF["Metric"] == metric_name]
    config = METRIC_LIBRARY[metric_name]

    empty_result = {
        "Metric": metric_name,
        "Category": config["category"],
        "Disclosure": 0,
        "Performance": 0,
        "Total": 0,
        "Weight": config["weight"],
        "Weighted Score": 0,
        "Evidence Count": 0,
        "Confidence": 0,
        "Disc_Mentioned": False,
        "Disc_Specific": False,
        "Disc_YoY": False,
        "Disc_Target": False,
        "Disc_Progress": False,
        "Perf_NetPositive": False,
        "Perf_Validation": False,
        "Perf_Trend": False,
        "Perf_Clean": False,
    }

    if evidence.empty:
        return empty_result

    full_text = " ".join(evidence["Paragraph"])

    # --------------------------------------------------------
    # Disclosure (0-5)
    # --------------------------------------------------------
    #
    # A "specific evidence" criterion defined only as "has a numeric
    # value" is structurally impossible for metrics explicitly
    # configured as qualitative (Human Rights, Anti-Corruption, ...),
    # capping their disclosure below quantitative metrics regardless of
    # how well they're actually documented. It's symmetric instead: a
    # numeric value for quantitative metrics, or a named
    # policy/framework/standard reference for qualitative ones.

    disc_mentioned = True

    if config["extract_numeric"]:
        metric_values = VALUES_DF[VALUES_DF["Metric"] == metric_name] if not VALUES_DF.empty else pd.DataFrame()
        disc_specific = not metric_values.empty
    else:
        disc_specific = contains_any(full_text, POLICY_REFERENCE_WORDS)

    metric_values = VALUES_DF[VALUES_DF["Metric"] == metric_name] if not VALUES_DF.empty else pd.DataFrame()
    has_dated_pair = (
        not metric_values.empty
        and metric_values["Year"].notna().sum() >= 2
        and metric_values.loc[metric_values["Year"].notna(), "Year"].nunique() >= 2
    )
    disc_yoy = contains_any(full_text, YOY_PHRASE_WORDS) or has_dated_pair

    disc_target = contains_any(full_text, TARGET_WORDS)
    disc_progress = contains_any(full_text, PROGRESS_WORDS)

    disclosure = sum([
        disc_mentioned, disc_specific, disc_yoy, disc_target, disc_progress
    ])

    # --------------------------------------------------------
    # Performance (0-5)
    # --------------------------------------------------------
    #
    # Sentiment is read through the metric's polarity (see
    # find_sentiment in Cell 4), so e.g. "increase the share of women"
    # counts as positive for Gender Diversity instead of being penalized
    # as a hit on a generic negative-word list. detect_verified_trend()
    # supplies the objective, data-driven Performance point.

    positives = evidence["Positive Words"].apply(len).sum()
    negatives = evidence["Negative Words"].apply(len).sum()

    perf_net_positive = positives > negatives and positives > 0
    perf_validation = contains_any(full_text, VALIDATION_WORDS)
    perf_trend = detect_verified_trend(metric_name, config)
    perf_clean = negatives == 0 and len(evidence) > 0

    performance = sum([
        perf_net_positive * 2,   # stronger signal, worth 2 points
        perf_validation,
        perf_trend,
        perf_clean,
    ])
    performance = min(performance, 5)

    total = disclosure + performance

    weighted = total / config["max_score"] * config["weight"]

    return {
        "Metric": metric_name,
        "Category": config["category"],
        "Disclosure": disclosure,
        "Performance": performance,
        "Total": total,
        "Weight": config["weight"],
        "Weighted Score": weighted,
        "Evidence Count": len(evidence),
        "Confidence": round(evidence["Confidence"].mean(), 1),
        "Disc_Mentioned": disc_mentioned,
        "Disc_Specific": disc_specific,
        "Disc_YoY": disc_yoy,
        "Disc_Target": disc_target,
        "Disc_Progress": disc_progress,
        "Perf_NetPositive": perf_net_positive,
        "Perf_Validation": perf_validation,
        "Perf_Trend": perf_trend,
        "Perf_Clean": perf_clean,
    }


# ------------------------------------------------------------
# Score Every Metric
# ------------------------------------------------------------

score_rows = [score_metric(metric) for metric in METRIC_LIBRARY]

METRIC_SCORE_DF = pd.DataFrame(score_rows)

print(METRIC_SCORE_DF[["Metric", "Category", "Disclosure", "Performance", "Total", "Weighted Score"]].to_string(index=False))
print()
print("Average metric score:", round(METRIC_SCORE_DF["Total"].mean(), 2))

## Cell 7 - Category & Overall Scoring

Retains the existing per-category calculation (`100 × weighted-score-sum / weight-sum`).
Overall = 70% Environmental + 20% Social + 10% Governance + the certification
mention bonus (capped at 10 points). Round to one decimal and cap at 100.
Shared rules in `scoring.py` assign A >80, B >70, C >55, D >25, otherwise F
(displayed as Starter). Utilities and questionnaires are excluded.
The 0.35 certificate boost and 80/20 materiality blend are deferred pending
validated certificate E/S/G values and industry mappings. See the revised
methodology documents in `docs/`. Evidence resolution and metric scoring
remain unchanged; accepted and excluded numeric evidence counts are retained.


In [ ]:
# ============================================================
# Cell 7 - ESG Category & Overall Scoring
# ============================================================
# Index aggregation uses 70/20/10 E/S/G weights and the retained mention bonus.

category_scores = (
    METRIC_SCORE_DF
    .groupby("Category")
    .apply(lambda x: 100 * x["Weighted Score"].sum() / x["Weight"].sum())
    .round(1)
)

CATEGORY_SCORE_DF = category_scores.rename("Score").reset_index()

# ------------------------------------------------------------
# Certification Bonus
# ------------------------------------------------------------

cert_bonus = min(len(CERTIFICATIONS_DF), MAX_CERTIFICATION_BONUS)

# ------------------------------------------------------------
# Overall Score
# ------------------------------------------------------------

overall_score = overall_from_categories(category_scores.to_dict(), cert_bonus)

overall_grade = grade_from_score(overall_score)

# ------------------------------------------------------------
# Data-quality note surfaced alongside the score
# ------------------------------------------------------------

n_needs_review = 0
n_values_total = len(VALUES_DF)

# ------------------------------------------------------------
# Store
# ------------------------------------------------------------

ESG_RESULTS = {
    "methodology_version": METHODOLOGY_VERSION,
    "overall_score": overall_score,
    "overall_grade": overall_grade,
    "category_scores": CATEGORY_SCORE_DF,
    "metric_scores": METRIC_SCORE_DF,
    "certification_bonus": cert_bonus,
    "values_needing_review": n_needs_review,
    "values_total": n_values_total,
    "candidate_values_total": len(CANDIDATE_VALUES_DF),
    "ambiguous_groups_excluded": len(VALUE_CONFLICTS_DF),
}

# ------------------------------------------------------------
# Console Output
# ------------------------------------------------------------

print("=" * 70)
print("FINAL ESG RESULTS")
print("=" * 70)
print()

print(f"Overall ESG Score   : {overall_score}/100")
print(f"Overall Grade       : {overall_grade}")
print(f"Certification Bonus : +{cert_bonus}")
if n_values_total:
    print(f"Accepted numeric values: {n_values_total} (0 require human review)")
print(f"Ambiguous value groups excluded automatically: {len(VALUE_CONFLICTS_DF)}")
print()

print(CATEGORY_SCORE_DF)
print()

print(
    METRIC_SCORE_DF
    .sort_values("Weighted Score", ascending=False)
    .to_string(index=False)
)

## Cell 8 - Primary KPI Selection

Every metric may have multiple extracted values; this picks one
representative KPI per metric for the dashboard.

- **`PREFERRED_UNITS`** lists, per metric, the canonical unit(s) to prefer
  when more than one is present (e.g. prefer `tCO2e` over a nearby `%` for
  Scope emissions) - each unit key here must match a canonical key in
  `UNIT_ALIASES` (Cell 1), not a raw spelling variant.
- **Cross-metric shared-source check**: if two or more different metrics
  end up resolving their KPI value from the *exact same* source
  paragraph, every one of them is flagged `Needs Review` - the report
  text doesn't let those metrics be told apart reliably. This is
  complementary to Cell 5.1: that cell catches one metric disagreeing
  with itself across different evidence; this catches two different
  metrics drawing from indistinguishable text.
- The selected row's `Source`/`Needs Review`/`Year` carry through, and an
  `Ambiguous` flag is set when more than one orphan-linked candidate
  exists for the same metric+unit, so the dashboard can be honest about
  which numbers are solid vs. reconstructed best-guesses instead of
  presenting everything with equal confidence.

In [ ]:
# ============================================================
# Cell 8 - Select Primary KPI Values
# ============================================================

"""
Every metric may have multiple extracted values; for the dashboard we
want one representative KPI per metric.

PREFERRED_UNITS points at the canonical unit keys the extractor
produces (Cell 1 normalizes unit spelling variants into one canonical
key, so this dict should never list two spellings of the same unit as
if they were different units it might find). Preferring the canonical
unit over a bare "%" is what stops a metric's KPI from silently
becoming an unrelated category-breakdown percentage.

The selected row's Source/Needs Review/Year carry through, and an
"Ambiguous" flag is set when more than one orphan-linked candidate
exists for the same metric+unit, so the dashboard can be honest about
which numbers are solid vs. reconstructed best-guesses instead of
presenting everything with equal confidence.
"""

# ------------------------------------------------------------
# Preferred Units
# ------------------------------------------------------------

PREFERRED_UNITS = {

    "Scope 1 Emissions": ["tCO2e", "%"],
    "Scope 2 Emissions": ["tCO2e", "%"],
    "Scope 3 Emissions": ["tCO2e", "%"],
    "Total GHG Emissions": ["tCO2e"],
    "Carbon Intensity": ["%", "tCO2e"],
    "Energy Consumption": ["GWh", "MWh", "kWh"],
    "Renewable Energy": ["%"],
    "Water Withdrawal": ["m3"],
    "Waste Generated": ["tonnes"],
    "Waste Recycled": ["%", "tonnes"],
    "Gender Diversity": ["%"],
    "Women in Leadership": ["%"],
    "Employee Turnover": ["%"],
    "Employee Engagement": ["%"],
    "Supplier Audits": ["%"],
    "Pay Gap": ["%"],
    "Independent Directors": ["%"],
    "Board Diversity": ["%"],
}

# ------------------------------------------------------------
# Select KPI
# ------------------------------------------------------------

primary_rows = []

for metric in METRIC_LIBRARY:

    metric_values = VALUES_DF[VALUES_DF["Metric"] == metric] if not VALUES_DF.empty else pd.DataFrame()

    if metric_values.empty:
        continue

    # Primary KPIs represent reported actuals. Numeric targets remain in
    # VALUES_DF for disclosure scoring but cannot displace an actual value.
    actual_values = metric_values[~metric_values["Target Context"]]
    if actual_values.empty:
        continue
    metric_values = actual_values

    # Scope 2 and total emissions are decision-useful on a market basis
    # when both accounting methods are disclosed; retain both in VALUES_DF.
    variant_order = ["market-based", "location-based", "default"] if metric in ("Scope 2 Emissions", "Total GHG Emissions") else ["default", "market-based", "location-based"]
    for variant in variant_order:
        variant_subset = metric_values[metric_values["Variant"] == variant]
        if len(variant_subset):
            metric_values = variant_subset
            break
    metric_values = metric_values.copy()
    metric_values["Current Report Year"] = metric_values["Year"].astype(str) == str(REPORT.get("report_year"))

    preferred = PREFERRED_UNITS.get(metric, [])
    selected = None
    selected_unit_pool = metric_values

    for unit in preferred:
        subset = metric_values[metric_values["Unit"] == unit]
        if len(subset):
            selected_unit_pool = subset
            selected = subset.sort_values(["Current Report Year", "Table Context", "Resolution Score", "Actual Context", "Year", "Confidence"], ascending=[False, False, False, False, False, False], na_position="last").iloc[0]
            break

    if selected is None:
        selected = metric_values.sort_values(["Current Report Year", "Table Context", "Resolution Score", "Actual Context", "Year", "Confidence"], ascending=[False, False, False, False, False, False], na_position="last").iloc[0]
        selected_unit_pool = metric_values[metric_values["Unit"] == selected["Unit"]]

    ambiguous = False

    primary_rows.append({
        "Category": selected["Category"],
        "Metric": metric,
        "Value": selected["Value"],
        "Unit": selected["Unit"],
        "Confidence": selected["Confidence"],
        "Source": selected["Source"],
        "Variant": selected.get("Variant", "default"),
        "Year Source": selected.get("Year Source"),
        "Table Context": bool(selected.get("Table Context", False)),
        "Needs Review": bool(selected["Needs Review"] or ambiguous),
        "Year": selected["Year"],
        "_Paragraph": selected["Paragraph"],
    })

PRIMARY_VALUES_DF = pd.DataFrame(primary_rows)

PRIMARY_VALUES_DF = PRIMARY_VALUES_DF.drop(columns=["_Paragraph"])



print("=" * 60)
print("PRIMARY KPI VALUES")
print("=" * 60)

print(PRIMARY_VALUES_DF.to_string(index=False))

if not PRIMARY_VALUES_DF.empty:
    n_flag = int(PRIMARY_VALUES_DF["Needs Review"].sum())
    if n_flag:
        print(f"\n{n_flag} of {len(PRIMARY_VALUES_DF)} KPI values are flagged 'Needs Review' "
              f"(reconstructed from table-style text where the value wasn't directly "
              f"adjacent to its unit) - verify these against the source before quoting them.")

## Batch evaluation - every report in `reports/`

When no `ESG_REPORT_PATH` override is supplied, this runs the analyzer for every
discovered `.txt` and `.pdf` and writes one auditable JSON result per company under
`output/analysis/`, plus `summary.json`. The dashboard cells below remain a separate
single-report preview and are not used by the batch evaluator.


In [ ]:
# Run the complete report folder without recursively invoking this cell.
if RUN_BATCH:
    import subprocess
    import sys

    batch_runner = Path("run_esg_reports.py").resolve()
    if not batch_runner.exists():
        raise FileNotFoundError(f"Batch runner not found: {batch_runner}")
    subprocess.run(
        [
            sys.executable,
            str(batch_runner),
            "--reports-dir", str(REPORTS_FOLDER),
            "--notebook", str(Path("esg_report_analyzer_v2.ipynb").resolve()),
            "--output-dir", str(Path("output/analysis").resolve()),
        ],
        check=True,
    )
else:
    print("Single-report mode: batch evaluation skipped.")
